In [0]:
# Load the Hotel Bookings Data
bookings_df = spark.table("workspace.default.hotel_bookings")

# Load the TripAdvisor Reviews Data
reviews_df = spark.table("workspace.default.tripadvisor_hotel_reviews")

display(bookings_df.limit(5))
display(reviews_df.limit(5))

In [0]:
%pip install faker

import pandas as pd
import numpy as np
from faker import Faker
import random

fake = Faker()

def generate_synthetic_spending(num_guests=5000, max_transactions_per_guest=5):
    """
    Generates synthetic cross-property spending logs for hotel guests.
    """
    categories = ['Casino', 'Spa', 'Fine Dining', 'Room Service', 'Gift Shop', 'Excursions']
    transactions = []

    # Generate a pool of mock Guest IDs to simulate a joinable key
    guest_ids = [f"GST-{10000 + i}" for i in range(num_guests)]
    
    for guest in guest_ids:
        # Not every guest spends extra money; random number of transactions
        num_transactions = random.randint(0, max_transactions_per_guest)
        
        for _ in range(num_transactions):
            category = random.choice(categories)
            
            # Create realistic spending amounts based on the category
            if category == 'Casino':
                amount = round(random.uniform(50.0, 5000.0), 2)
            elif category == 'Spa':
                amount = round(random.uniform(100.0, 450.0), 2)
            elif category == 'Fine Dining':
                amount = round(random.uniform(75.0, 350.0), 2)
            else:
                amount = round(random.uniform(15.0, 150.0), 2)
                
            transactions.append({
                'Transaction_ID': fake.uuid4(),
                'Guest_ID': guest,
                'Category': category,
                'Amount_USD': amount,
                'Transaction_Date': fake.date_between(start_date='-1y', end_date='today')
            })

    # Convert to Pandas DataFrame, then to a Spark DataFrame
    pdf = pd.DataFrame(transactions)
    return pdf

# Generate the data and convert to PySpark DataFrame
synthetic_pandas_df = generate_synthetic_spending(num_guests=1000)
synthetic_spark_df = spark.createDataFrame(synthetic_pandas_df)

display(synthetic_spark_df.limit(10))

In [0]:
# Save your Spark DataFrame as a permanent table
synthetic_spark_df.write.mode("overwrite").saveAsTable("default.synthetic_transactions")


In [0]:
from pyspark.sql.functions import monotonically_increasing_id, concat, lit


# add a unique Guest_ID to serve as our primary key
master_bookings_df = bookings_df.withColumn(
    "Guest_ID", 
    concat(lit("GST-"), monotonically_increasing_id())
)

# Display the new schema to verify
display(master_bookings_df.select("Guest_ID", "hotel", "is_canceled", "lead_time").limit(5))

In [0]:
from pyspark.sql.functions import rand, row_number
from pyspark.sql.window import Window

# Step 1: Count exactly how many reviews we have to map
review_count = reviews_df.count()
print(f"Total reviews to assign: {review_count}")

# Step 2: Shuffle the master bookings randomly and assign a temporary row number
window_spec_bookings = Window.orderBy(rand())
shuffled_bookings = master_bookings_df.withColumn("row_num", row_number().over(window_spec_bookings))

# Step 3: Assign a temporary row number to the reviews
window_spec_reviews = Window.orderBy(rand())
shuffled_reviews = reviews_df.withColumn("row_num", row_number().over(window_spec_reviews))

# Step 4: Map the reviews to a random selection of Guest_IDs
# We isolate the random Guest_IDs and join them directly to the reviews table
guest_review_mapping = shuffled_bookings.filter(shuffled_bookings.row_num <= review_count) \
    .select("Guest_ID", "row_num") \
    .join(shuffled_reviews, on="row_num", how="inner") \
    .drop("row_num")

# Step 5: The Master Join (Spoke 1 connects to the Hub)
# We LEFT JOIN the mapping back to the master hub so we don't lose any bookings!
final_hub_df = master_bookings_df.join(guest_review_mapping, on="Guest_ID", how="left")

# Display the results to verify! 
# Will see booking data for everyone, but review text only for the random subset.
display(final_hub_df.select("Guest_ID", "hotel", "is_canceled", "Rating", "Review").limit(15))

In [0]:
# Save the joined dataframe as a permanent Delta Table
final_hub_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("default.master_hotel_hub")

print("Successfully saved as a Delta table!")

In [0]:
import pandas as pd
import random
from faker import Faker

fake = Faker()

# 1. Grab 5,000 actual Guest_IDs from our saved Hub table
actual_guests_df = spark.sql("SELECT Guest_ID FROM default.master_hotel_hub LIMIT 5000")
# Convert to a standard Python list
real_guest_ids = [row.Guest_ID for row in actual_guests_df.collect()]

def generate_spending_for_real_guests(guest_list, max_transactions_per_guest=5):
    categories = ['Casino', 'Spa', 'Fine Dining', 'Room Service', 'Gift Shop', 'Excursions']
    transactions = []
    
    for guest in guest_list:
        # Not every guest spends extra money
        num_transactions = random.randint(0, max_transactions_per_guest)
        
        for _ in range(num_transactions):
            category = random.choice(categories)
            
            # Create realistic spending amounts based on the category
            if category == 'Casino':
                amount = round(random.uniform(50.0, 5000.0), 2)
            elif category == 'Spa':
                amount = round(random.uniform(100.0, 450.0), 2)
            elif category == 'Fine Dining':
                amount = round(random.uniform(75.0, 350.0), 2)
            else:
                amount = round(random.uniform(15.0, 150.0), 2)
                
            transactions.append({
                'Transaction_ID': fake.uuid4(),
                'Guest_ID': guest,
                'Category': category,
                'Amount_USD': amount,
                'Transaction_Date': fake.date_between(start_date='-1y', end_date='today')
            })

    return pd.DataFrame(transactions)

# 2. Generate the data using our REAL guest IDs
synthetic_pandas_df = generate_spending_for_real_guests(real_guest_ids)
synthetic_spark_df = spark.createDataFrame(synthetic_pandas_df)

# 3. Save this directly as our second Delta Table!
synthetic_spark_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("default.synthetic_spending_logs")

display(synthetic_spark_df.limit(5))